# 多路召回融合策略

基于 TIGER + Content last_1 两路 top-50 候选，探索融合策略。

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
from itertools import product

# 加载
tiger = np.load('../predictions/tiger_baseline_ce_top50.npy')
tiger_scores = np.load('../predictions/tiger_baseline_ce_top50_scores.npy')
content = np.load('../predictions/content_sim_last1_top50.npy')
content_scores = np.load('../predictions/content_sim_last1_top50_scores.npy')

test = pd.read_parquet('../data/Beauty/test.parquet')
targets = test['target'].values
histories = test['history'].values
N = len(test)

print(f"N={N}")
print(f"TIGER scores: [{tiger_scores[tiger_scores > -1e9].min():.3f}, {tiger_scores.max():.3f}]")
print(f"Content scores: [{content_scores[content_scores > 0].min():.3f}, {content_scores.max():.3f}]")

N=22363
TIGER scores: [-2.839, -0.004]
Content scores: [0.861, 0.999]


## 1. 评估工具

In [2]:
def evaluate_merged(merged_preds, targets, ks=[5, 10, 20]):
    """merged_preds: (N, max_k) int32 — 合并后的 top-k 列表"""
    results = {}
    for k in ks:
        hit = np.any(merged_preds[:, :k] == targets[:, None], axis=1)
        recall = hit.mean()
        match_pos = np.argmax(merged_preds[:, :k] == targets[:, None], axis=1)
        ndcg_vals = np.where(hit, 1.0 / np.log2(match_pos + 2.0), 0.0)
        results[f'Recall@{k}'] = recall
        results[f'NDCG@{k}'] = ndcg_vals.mean()
    return results

def print_metrics(metrics):
    for k in [5, 10, 20]:
        print(f"  Recall@{k}: {metrics.get(f'Recall@{k}', 0):.4f}  "
              f"NDCG@{k}: {metrics.get(f'NDCG@{k}', 0):.4f}")

# Baselines
print("=== Baselines ===")
print("TIGER:")
print_metrics(evaluate_merged(tiger, targets))
print("Content:")
print_metrics(evaluate_merged(content, targets))

# Upper bound: simple set union of top-20
union_k = np.zeros((N, 20), dtype=np.int32)
for i in range(N):
    seen = set()
    idx = 0
    for item in content[i, :20]:
        if item != 0 and item not in seen:
            seen.add(item)
            union_k[i, idx] = item
            idx += 1
    for item in tiger[i, :20]:
        if item != 0 and item not in seen and idx < 20:
            seen.add(item)
            union_k[i, idx] = item
            idx += 1
print("\nUpper bound (Content top-20 + TIGER top-20, Content first):")
print_metrics(evaluate_merged(union_k, targets))

=== Baselines ===
TIGER:
  Recall@5: 0.0346  NDCG@5: 0.0227
  Recall@10: 0.0566  NDCG@10: 0.0298
  Recall@20: 0.0835  NDCG@20: 0.0366
Content:
  Recall@5: 0.0439  NDCG@5: 0.0313
  Recall@10: 0.0589  NDCG@10: 0.0361
  Recall@20: 0.0779  NDCG@20: 0.0409

Upper bound (Content top-20 + TIGER top-20, Content first):
  Recall@5: 0.0439  NDCG@5: 0.0313
  Recall@10: 0.0589  NDCG@10: 0.0361
  Recall@20: 0.0779  NDCG@20: 0.0409


## 2. RRF 参数调优

扫 RRF 常数 k 和两路权重。`score = w_T/(k+rank_T) + w_C/(k+rank_C)`

In [3]:
def rrf_fusion(tiger_preds, content_preds, k_rrf=60, w_t=1.0, w_c=1.0, topk=20):
    """Weighted RRF merge."""
    merged = np.zeros((len(tiger_preds), topk), dtype=np.int32)
    for i in range(len(tiger_preds)):
        scores = defaultdict(float)
        for rank, item in enumerate(tiger_preds[i]):
            if item != 0:
                scores[item] += w_t / (k_rrf + rank + 1)
        for rank, item in enumerate(content_preds[i]):
            if item != 0:
                scores[item] += w_c / (k_rrf + rank + 1)
        top_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
        for j, (item, _) in enumerate(top_items):
            merged[i, j] = item
    return merged

# Sweep k
print("=== RRF k sweep (equal weights) ===")
best_k, best_r = 0, 0
for k_rrf in [0, 1, 5, 10, 20, 40, 60, 100, 200]:
    m = rrf_fusion(tiger, content, k_rrf=k_rrf)
    r = evaluate_merged(m, targets)
    print(f"k={k_rrf:3d}: R@5={r['Recall@5']:.4f}  R@10={r['Recall@10']:.4f}  R@20={r['Recall@20']:.4f}  N@20={r['NDCG@20']:.4f}")
    if r['Recall@20'] > best_r:
        best_k, best_r = k_rrf, r['Recall@20']
print(f"\nBest k={best_k} (R@20={best_r:.4f})")

=== RRF k sweep (equal weights) ===
k=  0: R@5=0.0523  R@10=0.0769  R@20=0.1101  N@20=0.0493
k=  1: R@5=0.0524  R@10=0.0771  R@20=0.1102  N@20=0.0497
k=  5: R@5=0.0540  R@10=0.0775  R@20=0.1110  N@20=0.0502
k= 10: R@5=0.0544  R@10=0.0785  R@20=0.1119  N@20=0.0504
k= 20: R@5=0.0532  R@10=0.0788  R@20=0.1125  N@20=0.0497
k= 40: R@5=0.0505  R@10=0.0766  R@20=0.1118  N@20=0.0485
k= 60: R@5=0.0504  R@10=0.0764  R@20=0.1118  N@20=0.0484
k=100: R@5=0.0501  R@10=0.0765  R@20=0.1118  N@20=0.0484
k=200: R@5=0.0503  R@10=0.0764  R@20=0.1118  N@20=0.0483

Best k=20 (R@20=0.1125)


In [4]:
# Sweep weights at best k
print(f"=== Weight sweep (k_rrf={best_k}) ===")
best_w_t, best_w_c, best_r = 1.0, 1.0, 0
for w_t, w_c in [(1.0, 0.5), (1.0, 0.25), (0.5, 1.0), (0.25, 1.0),
                 (1.5, 1.0), (1.0, 1.5), (2.0, 1.0), (1.0, 2.0),
                 (1.0, 1.0)]:
    m = rrf_fusion(tiger, content, k_rrf=best_k, w_t=w_t, w_c=w_c)
    r = evaluate_merged(m, targets)
    print(f"w_T={w_t:.2f} w_C={w_c:.2f}: R@5={r['Recall@5']:.4f}  R@10={r['Recall@10']:.4f}  R@20={r['Recall@20']:.4f}  N@20={r['NDCG@20']:.4f}")
    if r['Recall@20'] > best_r:
        best_w_t, best_w_c, best_r = w_t, w_c, r['Recall@20']
print(f"\nBest: w_T={best_w_t}, w_C={best_w_c} (R@20={best_r:.4f})")

=== Weight sweep (k_rrf=20) ===
w_T=1.00 w_C=0.50: R@5=0.0398  R@10=0.0623  R@20=0.0882  N@20=0.0400
w_T=1.00 w_C=0.25: R@5=0.0377  R@10=0.0601  R@20=0.0873  N@20=0.0390
w_T=0.50 w_C=1.00: R@5=0.0482  R@10=0.0652  R@20=0.0851  N@20=0.0444
w_T=0.25 w_C=1.00: R@5=0.0456  R@10=0.0615  R@20=0.0829  N@20=0.0432
w_T=1.50 w_C=1.00: R@5=0.0402  R@10=0.0622  R@20=0.1113  N@20=0.0460
w_T=1.00 w_C=1.50: R@5=0.0500  R@10=0.0660  R@20=0.1007  N@20=0.0487
w_T=2.00 w_C=1.00: R@5=0.0398  R@10=0.0623  R@20=0.0882  N@20=0.0400
w_T=1.00 w_C=2.00: R@5=0.0482  R@10=0.0652  R@20=0.0851  N@20=0.0444
w_T=1.00 w_C=1.00: R@5=0.0532  R@10=0.0788  R@20=0.1125  N@20=0.0497

Best: w_T=1.0, w_C=1.0 (R@20=0.1125)


## 3. Score 归一化融合

TIGER log-prob 和 Content cosine similarity 不在同一尺度。归一化后直接相加。

In [5]:
def score_fusion(tiger_preds, content_preds, tiger_scores, content_scores,
                 method='minmax', topk=20):
    """Score-based fusion with normalization."""
    merged = np.zeros((len(tiger_preds), topk), dtype=np.int32)
    for i in range(len(tiger_preds)):
        scores = defaultdict(float)
        # Collect raw scores for normalization
        t_items = []
        t_raw = []
        for item, s in zip(tiger_preds[i], tiger_scores[i]):
            if item != 0 and s > -1e9:
                t_items.append(item)
                t_raw.append(s)
        c_items = []
        c_raw = []
        for item, s in zip(content_preds[i], content_scores[i]):
            if item != 0 and s > 0:
                c_items.append(item)
                c_raw.append(s)
        
        if not t_raw and not c_raw:
            continue
        
        if method == 'minmax':
            t_min, t_max = min(t_raw) if t_raw else 0, max(t_raw) if t_raw else 1
            c_min, c_max = min(c_raw) if c_raw else 0, max(c_raw) if c_raw else 1
            t_range = t_max - t_min if t_max > t_min else 1
            c_range = c_max - c_min if c_max > c_min else 1
            for item, s in zip(t_items, t_raw):
                scores[item] += (s - t_min) / t_range
            for item, s in zip(c_items, c_raw):
                scores[item] += (s - c_min) / c_range
        elif method == 'rank':
            for rank, item in enumerate(t_items):
                scores[item] += 1.0 / (rank + 1)
            for rank, item in enumerate(c_items):
                scores[item] += 1.0 / (rank + 1)
        elif method == 'zscore':
            t_arr, c_arr = np.array(t_raw), np.array(c_raw)
            t_std, c_std = t_arr.std() if len(t_arr) > 1 else 1, c_arr.std() if len(c_arr) > 1 else 1
            t_mean, c_mean = t_arr.mean(), c_arr.mean()
            for item, s in zip(t_items, t_raw):
                scores[item] += (s - t_mean) / t_std
            for item, s in zip(c_items, c_raw):
                scores[item] += (s - c_mean) / c_std
        
        top_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
        for j, (item, _) in enumerate(top_items):
            merged[i, j] = item
    return merged

print("=== Score fusion methods ===")
for method in ['minmax', 'zscore', 'rank']:
    m = score_fusion(tiger, content, tiger_scores, content_scores, method=method)
    r = evaluate_merged(m, targets)
    print(f"\n{method}:")
    print_metrics(r)

=== Score fusion methods ===

minmax:
  Recall@5: 0.0549  NDCG@5: 0.0352
  Recall@10: 0.0789  NDCG@10: 0.0429
  Recall@20: 0.1114  NDCG@20: 0.0511

zscore:
  Recall@5: 0.0530  NDCG@5: 0.0353
  Recall@10: 0.0786  NDCG@10: 0.0435
  Recall@20: 0.1093  NDCG@20: 0.0512

rank:
  Recall@5: 0.0523  NDCG@5: 0.0329
  Recall@10: 0.0769  NDCG@10: 0.0409
  Recall@20: 0.1101  NDCG@20: 0.0493


## 4. 固定配额融合

Content top-A + TIGER top-B，去重（先到者保留）。两路各自内部已按分数排序。
测试：Content 优先 vs TIGER 优先 vs 交替。

In [6]:
def fixed_quota(tiger_preds, content_preds, a, b, order='content_first'):
    """Content top-a + TIGER top-b, dedup.
    order: 'content_first' | 'tiger_first' | 'interleave'"""
    topk = a + b
    merged = np.zeros((len(tiger_preds), topk), dtype=np.int32)
    for i in range(len(tiger_preds)):
        seen = set()
        idx = 0
        c_seq = content_preds[i]
        t_seq = tiger_preds[i]
        c_pos, t_pos = 0, 0
        
        if order == 'content_first':
            for _ in range(a):
                while c_pos < len(c_seq) and c_seq[c_pos] in seen:
                    c_pos += 1
                if c_pos < len(c_seq) and c_seq[c_pos] != 0:
                    seen.add(c_seq[c_pos])
                    merged[i, idx] = c_seq[c_pos]
                    idx += 1
                    c_pos += 1
            for _ in range(b):
                while t_pos < len(t_seq) and t_seq[t_pos] in seen:
                    t_pos += 1
                if t_pos < len(t_seq) and t_seq[t_pos] != 0:
                    seen.add(t_seq[t_pos])
                    merged[i, idx] = t_seq[t_pos]
                    idx += 1
                    t_pos += 1
        elif order == 'tiger_first':
            for _ in range(b):
                while t_pos < len(t_seq) and t_seq[t_pos] in seen:
                    t_pos += 1
                if t_pos < len(t_seq) and t_seq[t_pos] != 0:
                    seen.add(t_seq[t_pos])
                    merged[i, idx] = t_seq[t_pos]
                    idx += 1
                    t_pos += 1
            for _ in range(a):
                while c_pos < len(c_seq) and c_seq[c_pos] in seen:
                    c_pos += 1
                if c_pos < len(c_seq) and c_seq[c_pos] != 0:
                    seen.add(c_seq[c_pos])
                    merged[i, idx] = c_seq[c_pos]
                    idx += 1
                    c_pos += 1
        elif order == 'interleave':
            # Alternate, starting with the larger quota
            pairs = []
            max_rounds = max(a, b)
            for r in range(max_rounds):
                if r < a:
                    pairs.append(('c', r))
                if r < b:
                    pairs.append(('t', r))
            for src, _ in pairs:
                if src == 'c':
                    while c_pos < len(c_seq) and c_seq[c_pos] in seen:
                        c_pos += 1
                    if c_pos < len(c_seq) and c_seq[c_pos] != 0:
                        seen.add(c_seq[c_pos])
                        merged[i, idx] = c_seq[c_pos]
                        idx += 1
                        c_pos += 1
                else:
                    while t_pos < len(t_seq) and t_seq[t_pos] in seen:
                        t_pos += 1
                    if t_pos < len(t_seq) and t_seq[t_pos] != 0:
                        seen.add(t_seq[t_pos])
                        merged[i, idx] = t_seq[t_pos]
                        idx += 1
                        t_pos += 1
        merged[i, idx:] = 0
    return merged[:, :topk]

# Sweep (a,b) pairs for order='content_first' (best upper bound used this)
print("=== Fixed quota sweep (content_first) ===")
best_ab, best_r = '', 0
for a, b in [(5,15), (8,12), (10,10), (12,8), (15,5), (20,20)]:
    m = fixed_quota(tiger, content, a, b, order='content_first')
    r = evaluate_merged(m, targets)
    print(f"C{a}+T{b}: R@5={r['Recall@5']:.4f}  R@10={r['Recall@10']:.4f}  R@20={r['Recall@20']:.4f}  N@20={r['NDCG@20']:.4f}")
    if r['Recall@20'] > best_r:
        best_ab, best_r = f'C{a}+T{b}', r['Recall@20']
print(f"\nBest: {best_ab} (R@20={best_r:.4f})")

=== Fixed quota sweep (content_first) ===
C5+T15: R@5=0.0439  R@10=0.0761  R@20=0.1101  N@20=0.0504
C8+T12: R@5=0.0439  R@10=0.0703  R@20=0.1103  N@20=0.0496
C10+T10: R@5=0.0439  R@10=0.0589  R@20=0.1090  N@20=0.0489
C12+T8: R@5=0.0439  R@10=0.0589  R@20=0.1071  N@20=0.0482
C15+T5: R@5=0.0439  R@10=0.0589  R@20=0.1012  N@20=0.0465
C20+T20: R@5=0.0439  R@10=0.0589  R@20=0.0779  N@20=0.0409

Best: C8+T12 (R@20=0.1103)


In [7]:
# Compare ordering strategies at best (a,b)
print("=== Ordering comparison ===")
for order in ['content_first', 'tiger_first', 'interleave']:
    m = fixed_quota(tiger, content, 10, 10, order=order)
    r = evaluate_merged(m, targets)
    print(f"\n{order} (C10+T10):")
    print_metrics(r)

=== Ordering comparison ===

content_first (C10+T10):
  Recall@5: 0.0439  NDCG@5: 0.0313
  Recall@10: 0.0589  NDCG@10: 0.0361
  Recall@20: 0.1090  NDCG@20: 0.0489

tiger_first (C10+T10):
  Recall@5: 0.0346  NDCG@5: 0.0227
  Recall@10: 0.0566  NDCG@10: 0.0298
  Recall@20: 0.1087  NDCG@20: 0.0434

interleave (C10+T10):
  Recall@5: 0.0520  NDCG@5: 0.0348
  Recall@10: 0.0762  NDCG@10: 0.0426
  Recall@20: 0.1091  NDCG@20: 0.0509


## 5. 历史长度分桶动态配额

唯一可用的 inference 端信号。每个长度桶给不同的 (C, T) 配额。
同时也测试「硬选择」：每个桶只用其中一路。

In [8]:
hist_lens = np.array([len(list(h)) for h in histories])
lo_len = int(np.percentile(hist_lens, 33.3))
hi_len = int(np.percentile(hist_lens, 66.7))
print(f"Buckets: short ≤{lo_len}, medium {lo_len+1}-{hi_len}, long >{hi_len}")

mask_short = hist_lens <= lo_len
mask_medium = (hist_lens > lo_len) & (hist_lens <= hi_len)
mask_long = hist_lens > hi_len
print(f"short: {mask_short.sum()}, medium: {mask_medium.sum()}, long: {mask_long.sum()}")

Buckets: short ≤5, medium 6-7, long >7
short: 11384, medium: 4490, long: 6489


In [9]:
def dynamic_quota_by_mask(tiger_preds, content_preds, masks, quotas, topk=20):
    """masks: list of boolean arrays; quotas: list of (a, b) tuples — one per mask."""
    merged = np.zeros((len(tiger_preds), topk), dtype=np.int32)
    for mask, (a, b) in zip(masks, quotas):
        indices = np.where(mask)[0]
        # Build merged for this bucket
        bucket_merged = fixed_quota(
            tiger_preds[indices], content_preds[indices], a, b, order='content_first'
        )[:, :topk]
        merged[indices] = bucket_merged
    return merged

# From the analysis notebook:
# Short: Content better at all k → more Content quota
# Medium: mixed
# Long: TIGER better at R@20 → more TIGER quota

print("=== Dynamic quota experiments ===")

configs = [
    # (short_a, short_b), (med_a, med_b), (long_a, long_b), label
    ((12, 8),  (10, 10), (8, 12),  'content-heavy short, balanced mid, tiger-heavy long'),
    ((15, 5),  (12, 8),  (8, 12),  'extreme content short, tiger long'),
    ((15, 5),  (10, 10), (5, 15),  'very extreme'),
    ((20, 0),  (10, 10), (0, 20),  'hard-selection: content short, tiger long'),
    ((20, 0),  (20, 0),  (0, 20),  'hard-selection: content short+mid, tiger long'),
]

for q_short, q_med, q_long, label in configs:
    masks = [mask_short, mask_medium, mask_long]
    quotas = [q_short, q_med, q_long]
    m = dynamic_quota_by_mask(tiger, content, masks, quotas)
    r = evaluate_merged(m, targets)
    print(f"\n{label}")
    print(f"  S{q_short} M{q_med} L{q_long}:"
          f"  R@5={r['Recall@5']:.4f}  R@10={r['Recall@10']:.4f}  R@20={r['Recall@20']:.4f}  N@20={r['NDCG@20']:.4f}")

=== Dynamic quota experiments ===

content-heavy short, balanced mid, tiger-heavy long
  S(12, 8) M(10, 10) L(8, 12):  R@5=0.0439  R@10=0.0628  R@20=0.1093  N@20=0.0490

extreme content short, tiger long
  S(15, 5) M(12, 8) L(8, 12):  R@5=0.0439  R@10=0.0628  R@20=0.1070  N@20=0.0482

very extreme
  S(15, 5) M(10, 10) L(5, 15):  R@5=0.0439  R@10=0.0646  R@20=0.1082  N@20=0.0489

hard-selection: content short, tiger long
  S(20, 0) M(10, 10) L(0, 20):  R@5=0.0417  R@10=0.0596  R@20=0.0890  N@20=0.0423

hard-selection: content short+mid, tiger long
  S(20, 0) M(20, 0) L(0, 20):  R@5=0.0417  R@10=0.0596  R@20=0.0823  N@20=0.0405


In [10]:
# Grid search over quotas per bucket (coarse to keep runtime reasonable)
print("=== Grid search (coarse) ===")
candidates = [(15,5), (12,8), (10,10), (8,12), (5,15)]
best_overall, best_r = None, 0
for q_short in candidates:
    for q_med in candidates:
        for q_long in candidates:
            m = dynamic_quota_by_mask(tiger, content,
                                       [mask_short, mask_medium, mask_long],
                                       [q_short, q_med, q_long])
            r = evaluate_merged(m, targets)['Recall@20']
            if r > best_r:
                best_r = r
                best_overall = (q_short, q_med, q_long)
print(f"Best: S{best_overall[0]} M{best_overall[1]} L{best_overall[2]} → R@20={best_r:.4f}")

=== Grid search (coarse) ===
Best: S(8, 12) M(5, 15) L(5, 15) → R@20=0.1110


## 6. 汇总结论

In [11]:
print("=" * 60)
print("FUSION SUMMARY")
print("=" * 60)

print("\n--- Baselines ---")
for name, preds in [('TIGER', tiger), ('Content', content)]:
    r = evaluate_merged(preds, targets)
    print(f"{name}: R@5={r['Recall@5']:.4f} R@10={r['Recall@10']:.4f} R@20={r['Recall@20']:.4f} N@20={r['NDCG@20']:.4f}")

print("\n--- Upper bound ---")
r_ub = evaluate_merged(union_k, targets)
print(f"Content20+Tiger20 (Content first): R@20={r_ub['Recall@20']:.4f} N@20={r_ub['NDCG@20']:.4f}")

print("\n--- Best RRF ---")
m_best_rrf = rrf_fusion(tiger, content, k_rrf=best_k, w_t=best_w_t, w_c=best_w_c)
r_best_rrf = evaluate_merged(m_best_rrf, targets)
print(f"k={best_k} w_T={best_w_t} w_C={best_w_c}: R@20={r_best_rrf['Recall@20']:.4f} N@20={r_best_rrf['NDCG@20']:.4f}")

print("\n--- Best score fusion ---")
for method in ['minmax', 'zscore', 'rank']:
    m = score_fusion(tiger, content, tiger_scores, content_scores, method=method)
    r = evaluate_merged(m, targets)
    print(f"{method}: R@20={r['Recall@20']:.4f} N@20={r['NDCG@20']:.4f}")

print("\n--- Best fixed quota ---")
print(f"{best_ab}: R@20={best_r:.4f}")
for order in ['content_first', 'tiger_first', 'interleave']:
    m = fixed_quota(tiger, content, 10, 10, order=order)
    r = evaluate_merged(m, targets)
    print(f"C10+T10 {order}: R@20={r['Recall@20']:.4f} N@20={r['NDCG@20']:.4f}")

print("\n--- Best dynamic quota (history length) ---")
if best_overall:
    m = dynamic_quota_by_mask(tiger, content,
                               [mask_short, mask_medium, mask_long],
                               list(best_overall))
    r_dyn = evaluate_merged(m, targets)
    print(f"S{best_overall[0]} M{best_overall[1]} L{best_overall[2]}: R@20={r_dyn['Recall@20']:.4f} N@20={r_dyn['NDCG@20']:.4f}")

FUSION SUMMARY

--- Baselines ---
TIGER: R@5=0.0346 R@10=0.0566 R@20=0.0835 N@20=0.0366
Content: R@5=0.0439 R@10=0.0589 R@20=0.0779 N@20=0.0409

--- Upper bound ---
Content20+Tiger20 (Content first): R@20=0.0779 N@20=0.0409

--- Best RRF ---
k=20 w_T=1.0 w_C=1.0: R@20=0.1125 N@20=0.0497

--- Best score fusion ---
minmax: R@20=0.1114 N@20=0.0511
zscore: R@20=0.1093 N@20=0.0512
rank: R@20=0.1101 N@20=0.0493

--- Best fixed quota ---
C8+T12: R@20=0.1110
C10+T10 content_first: R@20=0.1090 N@20=0.0489
C10+T10 tiger_first: R@20=0.1087 N@20=0.0434
C10+T10 interleave: R@20=0.1091 N@20=0.0509

--- Best dynamic quota (history length) ---
S(8, 12) M(5, 15) L(5, 15): R@20=0.1110 N@20=0.0503


## 7. Full NDCG for best configs

补打最优配置的完整 NDCG（N@5, N@10, N@20）。

In [ ]:
print("=== Full NDCG for best configs ===\n")

# Best RRF
print(f"RRF (k={best_k}, w_T={best_w_t}, w_C={best_w_c}):")
m_rrf = rrf_fusion(tiger, content, k_rrf=best_k, w_t=best_w_t, w_c=best_w_c)
print_metrics(evaluate_merged(m_rrf, targets))

# Best fixed quota
a, b = int(best_ab[1]), int(best_ab[4:])  # Parse "C8+T12"
print(f"\nFixed quota {best_ab} (content_first):")
m_fq = fixed_quota(tiger, content, a, b, order='content_first')
print_metrics(evaluate_merged(m_fq, targets))

# Best dynamic quota
if best_overall:
    print(f"\nDynamic quota S{best_overall[0]} M{best_overall[1]} L{best_overall[2]}:")
    m_dyn = dynamic_quota_by_mask(tiger, content,
                                   [mask_short, mask_medium, mask_long],
                                   list(best_overall))
    print_metrics(evaluate_merged(m_dyn, targets))